# Player-Race MMR by Date

Load the daily player-race MMR aggregation and plot average MMR trends for the most active player-race combinations.

In [1]:
import pandas as pd
from analysis import load_replays, build_player_race_mmr_by_day

replays = load_replays()
rows = build_player_race_mmr_by_day(replays)
df = pd.DataFrame(rows)
df['day'] = pd.to_datetime(df['day'])
df = df.sort_values(['player_name', 'race', 'day'])

df = df.loc[(df['race'] == 'Z') & (df['avg_mmr'] >= 2500)]

player_game_counts = df.groupby('player_name')['replay_count'].sum()
active_players = player_game_counts[player_game_counts > 10].index

df = df[df['player_name'].isin(active_players)]

df.head()


,player_name,race,day,replay_count,avg_mmr,min_mmr,max_mmr,wins,losses
1215,Ali,Z,2026-06-12,0,5028.0,5028.0,5028.0,0,0
1216,Ali,Z,2026-06-13,0,5028.0,5028.0,5028.0,0,0
1217,Ali,Z,2026-06-14,0,5028.0,5028.0,5028.0,0,0
1218,Ali,Z,2026-06-15,0,5028.0,5028.0,5028.0,0,0
1219,Ali,Z,2026-06-16,0,5028.0,5028.0,5028.0,0,0


In [5]:
import plotly.express as px

# Plot each Z player as a separate line with markers and hover labels
fig = px.line(
    df,
    x='day',
    y='avg_mmr',
    color='player_name',
    line_group='player_name',
    markers=True,
    hover_data=['race', 'replay_count', 'min_mmr', 'max_mmr', 'wins', 'losses'],
    title='Player-Race Daily Average MMR (Z only)',
)
fig.update_traces(opacity=0.35)

# Add a thick group average line on top
group_avg = df.groupby('day', as_index=False)['avg_mmr'].mean()
fig.add_scatter(
    x=group_avg['day'],
    y=group_avg['avg_mmr'],
    mode='lines',
    line={'color': 'black', 'width': 4},
    name='Group average',
    hovertemplate='day=%{x}<br>avg_mmr=%{y:.1f}<extra></extra>',
)

fig.update_layout(
    xaxis_title='Date',
    yaxis_title='Average MMR',
    legend_title='Player',
    hovermode='closest',
    template='plotly_white',
    legend=dict(itemclick='toggleothers', itemdoubleclick='toggle'),
    height=1200,
    width=1600,
)

fig.show()
